# GEMS Geological Data Atlas V1
### Data acquisition, geological feature engineering and fault-discovery EDA

**Question.** Where do the available geophysical and topographic measurements indicate potentially unmapped geological structures, at what spatial scales, with what geological evidence, and with what limitations?

## 00 - Executive overview

**Competition problem.** The DrivenData GEMS challenge asks for maps of faults that are *not yet mapped*, scored by a distance-weighted Tversky index on a 100 m grid (EPSG:32611). This notebook does **not** train a model or produce a submission: it builds the data foundation and describes the geological evidence.

**Data sources.** (1) GeoDAWN numerical raster - magnetic, gravity, terrain, basement, conductivity, strain and seismicity bands at 100 m; (2) mapped faults (`existing_faults.tif`), an incomplete reference; (3) the official template; (4) USGS 3DEP 1 m DEM tiles where they were downloaded.

**Workspace.** Everything is written to `GEMS_2/GEMS_GEOLOGICAL_DATA_ATLAS_V1/`. After a verified raw snapshot (section 01) every stage reads only that snapshot and this atlas' own products. Each stage is fingerprinted and resumable: after a disconnect, run section 00 again, then the section you were in.

**Reading guide.** Each section states its geological question, method and assumptions; shows figures; prints the measured numbers; gives an interpretation generated from those numbers (OBSERVATION vs HYPOTHESIS); and lists its limitations.

In [ ]:
CONFIG = {
    "drive_root": "/content/drive/MyDrive/GEMS_2",
    "atlas_root": "GEMS_GEOLOGICAL_DATA_ATLAS_V1",

    "snapshot_raw": True,
    "verify_checksums": True,

    "enable_new_dem_downloads": False,
    "dem_mode": "FAST_10M",          # "DETAILED_2M" only when explicitly wanted (much slower)

    "run_ingestion": True,
    "run_feature_engineering": True,
    "run_eda": True,

    "reuse_valid_cache": True,
    "save_figures": True,
    "export_csv": True,
    "export_parquet": True,
    "export_geotiff": True,

    "seed": 2026,
}
REPO, REF, CODE = "mybenkhadda/GEMS---drivendata", "main", "/content/GEMS_ATLAS"

In [ ]:
import subprocess, sys
from pathlib import Path
from google.colab import drive
from IPython.display import display, Markdown

drive.mount("/content/drive")
if not (Path(CODE) / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, f"https://github.com/{REPO}.git", CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "fetch", "--depth", "1", "origin", REF], check=True)
    subprocess.run(["git", "-C", CODE, "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE + "[colab]", "plotly"], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
HEAD = subprocess.check_output(["git", "-C", CODE, "rev-parse", "HEAD"], text=True).strip()

from gems_bench.geological_atlas import AtlasPipeline, GeologicalAtlas
from gems_bench.geological_atlas import figures as F, reports as R, visualization as V
V.style()
P = AtlasPipeline(CONFIG)
show = lambda key: display(Markdown(R.section(P.results, key)))
print("code commit:", HEAD, "| atlas:", P.paths.root)

## 01 - Acquisition and raw snapshot

**Question.** Which original raw inputs exist, are they readable, and do we hold a verified copy?

**Method.** A Drive health check (stat, list, write-fsync-read-delete probe) runs first. Discovery scans only the original raw folders (`data/raw`, `data/raw/geodawn`, `data/raw/faults`, `data/raw/dem_1m/**`, catalog JSON, licence / provenance documents). The preflight table is built from those real files. The snapshot copies each file in 16 MiB chunks to `<file>.partial`, hashes the source stream and the re-read copy, and renames only on equal SHA-256; a ledger makes it resumable and stops re-hashing verified files.

**Assumptions / safeguards.** Originals are read-only. `Errno 107` stops the stage as a Drive failure (never "missing" or "corrupt"); quota errors stop with an incomplete-copy report; an existing verified copy is never overwritten by an incomplete one.

In [ ]:
print(P.health())
SOURCES = P.discover()
display(SOURCES.groupby(["role", "access_status"]).agg(files=("source_id", "count"), GiB=("size_bytes", lambda s: s.astype(float).sum() / 2**30)))
PRE = P.preflight(SOURCES)
display(PRE)

In [ ]:
import pandas as pd
if CONFIG["run_ingestion"]:
    SNAP = P.snapshot(SOURCES)
    cap = SNAP["capacity"]
    print(f"sources {cap['sources']}, total {cap['total_source_bytes']/2**30:.2f} GiB, to copy {cap['bytes_to_copy']/2**30:.2f} GiB, "
          f"reported free {(cap['destination_free_bytes'] or 0)/2**30:.1f} GiB (fits: {cap['fits']})")
    print("status counts:", SNAP["counts"], "| complete:", SNAP["complete"])
    display(pd.read_csv(P.paths.inventory / "provenance.csv").groupby(["role", "snapshot_status"]).size().rename("files"))

## 02 - Study area and data availability

**Question.** What is the analysis grid, where is each modality valid, and how much of the area does the high-resolution DEM really cover?

**Method.** The GeoDAWN raster defines the grid (EPSG:32611, 100 m; the template must match exactly). Valid numerical footprint: >= 50 % of bands valid. Labels on another grid are reprojected nearest-neighbour. DEM tiles are classified from their headers and a decimated data read (VALID, UNREADABLE, OUTSIDE_STUDY_AREA, UNKNOWN_VERTICAL_UNIT, DUPLICATE_OVERLAP, CATALOG_ONLY). Coverage = area-weighted share of valid DEM pixels per 100 m cell (GDAL `average` of the validity mask), not tile rectangles.

**Optional acquisition** (`enable_new_dem_downloads`): label-free, coverage-first selection from the TNM catalog with tile / GiB budgets, resumable downloads into the snapshot.

In [ ]:
print(P.geospatial())
DEM_INV = P.dem_inventory()
display(DEM_INV.groupby("status").agg(tiles=("file", "count")))
if CONFIG["enable_new_dem_downloads"]:
    display(P.acquire())
    DEM_INV = P.dem_inventory()        # re-inventory with the new tiles (fingerprint includes the download ledger)
print(P.results["dem_inventory"])
F.study_area(P); F.dem_coverage(P)

**Limitations.** Coverage is measured from a 10 m decimated read; sub-10 m holes are counted at that resolution. CATALOG_ONLY tiles are listed, not data. A DEM tile whose vertical unit cannot be established is excluded until `dem.vertical_unit_override` is set by you after reading its metadata.

## 03 - Raw geophysical atlas

**Question.** What does each delivered band contain - identifier, modality, unit, valid area, value range?

**Method.** The band registry is read from the real raster metadata (band tags / descriptions); units come from metadata when present, otherwise from documented defaults and are marked as such. Every band is exported as a GeoTIFF (nodata -> NaN) into `03_features/raw_bands/`.

**Caution.** Total-field products (`tmi`, `mag_anom`) are not reduced to the pole and are never treated as RTP.

In [ ]:
P.raw_bands()
display(pd.read_csv(P.paths.root / "03_features/raw_bands/raw_band_registry.csv")[["band_index", "band_name", "modality", "unit", "unit_source", "valid_fraction_sampled", "p01", "p50", "p99"]])
F.band_atlas(P)

## 04 - Geological signatures of mapped faults

**Question.** How long, how fragmented, how oriented and how unevenly distributed is the mapped fault network?

**Method.** Skeleton of the label raster; trace length = 8-connected link length (100 m / 141 m); trace strike from the principal axis of each trace (axial, 0-180 deg from north); length-weighted axial mean and resultant R; Rayleigh test on doubled angles; density per 3 x 3 subregion.

**Assumption.** Mapped faults are context and diagnostics only - an incomplete map, not ground truth for undiscovered structures.

In [ ]:
P.faults()
F.fault_network(P)
show("faults")

## 05 - Magnetic field interpretation

**Question.** Which magnetic boundaries exist, how coherent are they, and do they persist from shallow to deep source levels?

**Method.** From the RTP band only: FFT derivatives with angular wavenumbers (nT/m), padded (25 % reflect) and cosine-tapered; THD, vertical derivative, tilt, analytic signal, second horizontal derivatives, boundary strike; upward continuation to 0.5 / 1 / 2 / 5 km, residuals, THD of each continued field; per-level THD percentile ranks -> boundary persistence, cross-scale stability, boundary strength, edge density; structure-tensor coherence, anisotropy and strike at 300 m / 1 km / 3 km and their consistency.

**Assumptions.** Edge guard of 1 km (continued fields: min(h, 2 km)) against padding artefacts. Not every gradient is a fault.

In [ ]:
if CONFIG["run_feature_engineering"]:
    P.magnetic()
F.field_maps(P, "mag")
show("magnetic")

## 06 - Gravity field interpretation

**Question.** Where are the density boundaries, how do their scales compare with the magnetic ones, and do the delivered gradient bands duplicate what we compute?

**Method.** Same operators as section 05 on `iso_grav_anom` (mGal). The recomputed THD / VD are compared with the delivered `_hg` / `_vg` bands (Spearman, block sample); a duplicate is not stored twice (registry status `alias_of_delivered`). Radial power spectra and energy per wavelength band compare magnetic and gravity scale content.

**Limitation.** No source depth is estimated from spectrum slopes.

In [ ]:
if CONFIG["run_feature_engineering"]:
    P.gravity()
F.field_maps(P, "grav"); F.spectra(P); F.continuation(P)
p = P.paths.d("04_eda", "tables") / "gravity_delivered_vs_recomputed.csv"
if p.exists(): display(pd.read_csv(p))
show("gravity")

## 07 - Regional terrain structure

**Question.** Which range fronts, ridges, valleys and slope breaks organise the 100 m topography, and along which strikes?

**Method.** From `det_elev` at 300 m, 1, 2, 3 km (sigma = scale / 2): slope, profile and plan curvature, ridge / valley strength (Hessian), TPI, local relief, roughness, slope breaks, structure-tensor coherence and strike, directional gradients, difference-of-Gaussians.

**Limitation.** 100 m data cannot resolve metre-scale scarps; that is section 08.

In [ ]:
if CONFIG["run_feature_engineering"]:
    P.terrain()
F.terrain(P)
show("terrain")

## 08 - High-resolution USGS DEM

**Question.** Where the 1 m DEM exists, which fine topographic structures (scarps, slope breaks, aligned edges) does it show, and how do 10 m and 2 m processing compare?

**Method.** FAST_10M (default): each tile is averaged to a 10 m grid aligned with the analysis grid (GDAL `average`, unit -> metres) and cached once; 5 km windows with a 1.2 km halo are mosaicked by deterministic source preference; seams between DEM projects are masked (60 m). Operators: slope, curvatures, TPI 100 / 500 m, relief, 4-azimuth edges, structure tensor 50 / 200 m, scarp step, ridge / valley, roughness. DETAILED_2M (opt-in, and always for 3 example windows): 1 m -> 2 m, 8 azimuths, structure tensor 5 / 20 m, TPI 50 / 200 m, crest / toe. Aggregation to 100 m with exact area weights, per-feature support fractions and measured coverage.

**Runtime.** The longest CPU stage; resumable per window (`dem.budget_minutes` > 0 stops cleanly - re-run the cell).

In [ ]:
if CONFIG["run_feature_engineering"]:
    DEM = P.dem()
    print(DEM)
    if DEM.get("status") == "INCOMPLETE":
        raise RuntimeError(f"DEM windows left: {DEM['windows_left']} - re-run this cell (finished windows are kept)")
    if DEM.get("status") not in ("NO_DEM",):
        print(P.dem_examples())
F.dem_maps(P); F.dem_examples(P)
show("dem")

## 09 - Multiscale and directional geology

**Question.** Do boundaries persist across scales, and do the modalities share a structural grain?

**Method.** Persistence / stability maps from section 05-06; coherence-weighted axial strike roses per modality (block sample: one pixel per 1 km block) next to the length-weighted mapped-fault rose. Axial statistics throughout - never arithmetic means of strikes.

In [ ]:
F.persistence(P)
display(F.strike_roses(P))

## 10 - Cross-modal geological agreement

**Question.** Where do independent measurements put a boundary at the same place with the same strike?

**Method.** Three independent groups - magnetic, gravity, topography (100 m terrain and DEM count as ONE group). Edge agreement sqrt(rank_a x rank_b); strike agreement cos 2(da) where both coherences >= 0.3; evidence count = groups with rank >= 0.9; available groups recorded separately so missing DEM reduces evidence instead of creating disagreement; continuity along strike (+-1 km); disagreement where one group is strong and another weak.

**Caution.** These are descriptive evidence layers, not fault probabilities.

In [ ]:
if CONFIG["run_feature_engineering"]:
    print(P.cross_modal())
F.orientation_agreement(P); F.boundary_agreement(P)
show("cross_modal")

## 11 - Fault proximity and feature response

**Question.** How do the features behave near mapped faults compared with far from them?

**Method.** Distance bands 0-300 m, 0.3-1 km, 1-3 km, > 3 km; one random pixel per 1 km block per band (adjacent pixels are not independent); medians, quantiles, near-far contrast in units of the far-band IQR; response curves; fault-normal profiles across long, linear, training-visible mapped traces (one per subregion).

**Caution.** Descriptive associations with an incomplete map - not causal, not used to select features or tune the lineament rules.

In [ ]:
if CONFIG["run_eda"]:
    BANDS = P.proximity(); P.profiles()
F.response(P); F.profiles(P)
show("proximity")

## 12 - Feature quality, uniqueness and redundancy

**Question.** Which features carry distinct, spatially coherent information, and which are redundant, noisy, limited or artefact-driven?

**Method (label-free).** Valid fraction, median / IQR / MAD, lag-1 spatial autocorrelation, correlation with distance to the footprint edge, DEM seam / non-seam ratio, adjacent-scale consistency, Spearman matrix on a block sample, average-linkage redundancy clusters (|rho| >= 0.9) with one representative each.

In [ ]:
if CONFIG["run_eda"]:
    Q = P.quality()
    display(Q["class"].value_counts())
F.redundancy(P)
show("quality")

## 13 - Candidate unmapped structural lineaments

**Question.** Where does the combined evidence trace linear structures, and how many lie away from mapped faults?

**Method (fixed before looking at faults).** Combined evidence >= its 92nd percentile, skeleton, junctions removed, segments >= 3 km with linearity >= 0.85. Each candidate carries length, strike, linearity, continuity, dominant scale, per-modality evidence, strike agreement, persistence, DEM coverage quality, seam proximity, evidence count, interpretation and limitations. `mapped_fault_distance_m` is a diagnostic annotation added afterwards.

**Status.** Every candidate is an **unverified geological structure**.

In [ ]:
if CONFIG["run_eda"]:
    LIN = P.lineaments()
else:
    LIN = GeologicalAtlas.open(P.paths.root).load_lineaments()
F.lineament_map(P, LIN); F.evidence_summary(P, LIN)
if LIN is not None and len(LIN):
    display(LIN.drop(columns="geometry").sort_values(["evidence_count", "length_km"], ascending=False).head(20))
show("lineaments")

In [ ]:
# Optional interactive exploration (downsampled previews; the static figures above remain the reference)
try:
    F.interactive(P, LIN).show()
except Exception as exc:
    print("interactive map unavailable:", exc)

## 14 - Model-ready data foundation

Retained features (section 12) plus every axial orientation component are stacked per family into multi-band GeoTIFFs (`06_ml_ready/feature_stack/`), with separate coverage masks, label-free robust normalisation (`06_ml_ready/normalization/robust_quantiles.json`) and a schema. Fault labels are not part of the stack. The reload API below reads everything without recomputation.

In [ ]:
SCHEMA = P.ml_ready()
display(pd.DataFrame([(k, len(v["bands"]), v["file"]) for k, v in SCHEMA["families"].items()], columns=["family", "bands", "file"]))
atlas = GeologicalAtlas.open(P.paths.root)
features = atlas.load_features(families=["magnetic", "gravity", "terrain", "dem"])
coverage = atlas.load_coverage()
print(len(features), "features reloaded; coverage layers:", list(coverage))

## 15 - Geological findings and limitations

Generated from the stored numbers only (`08_reports/`). OBSERVATION = measured; HYPOTHESIS = geological reading of a measurement.

In [ ]:
P.write_reports()
for f in ("geological_findings.md", "feature_recommendations.md", "data_limitations.md"):
    display(Markdown((P.paths.reports / f).read_text()))

## 16 - Artifact inventory and handoff

Every stage record (fingerprint, outputs, status) and every file of the atlas (snapshot excluded) with its size. Reload paths: `GeologicalAtlas.open(...)`; processing parameters: `08_reports/run_manifest.json`; docs: `docs/geological_atlas/`.

In [ ]:
STAGES = P.artifact_report()
display(STAGES)
inv = pd.read_csv(P.paths.reports / "artifact_inventory.csv")
display(inv.groupby("section").agg(files=("path", "count"), GiB=("bytes", lambda s: s.sum() / 2**30)))
print("figures saved this session:", len(P.figs))